# Módulo 1 — Referencias poblacionales de envejecimiento

Atributos clásicos, ICC test-retest y curvas longitudinales TRAIN. Este cuaderno no entrena difusión ni genera imágenes. FFHQ-Aging es un proxy del dominio de entrenamiento, no una etiqueta de nacionalidad. `sample_kaggle` no es FFHQ-Aging.


In [ ]:
# Ejecuta manualmente en Colab, desde la raíz del repositorio.
%pip install -e ".[aging-bias]"


In [ ]:
from pathlib import Path
import pandas as pd
from src.aging_bias import ReferenceConfig, FeatureConfig, build_reference_bundle, describe_population_aging, run_self_checks
from src.comparisong.evaluation_backends_v2 import prepare_reference_backend

COLOMBIA_TRAIN = Path("/content/drive/MyDrive/dataset_def/train")
FFHQ_AGING = Path("/content/drive/MyDrive/FFHQ-Aging")
TEST_MANIFEST = Path("/content/drive/MyDrive/respaldo_test_comparison/transitions.csv")
REFERENCE_DIR = Path("/content/drive/MyDrive/aging_bias_references")
ASSETS = Path("/content/evaluation_models_v2")


## 1. Validación matemática, sin redes

Se detiene si la regresión o los pesos no cumplen las anclas sintéticas.


In [ ]:
checks = run_self_checks(REFERENCE_DIR / "validation")
display(checks)
assert checks.passed.all()


## 2. Preparar solo landmarks (CPU)

Descarga explícita al ejecutar esta celda: un FaceLandmarker task pequeño. No carga FairFace, AdaFace, DEX ni Inception. Archivos existentes se reutilizan.


In [ ]:
reference_backend = prepare_reference_backend(ASSETS, feature_config=FeatureConfig(crop_size=256, harmonize_sigma=1.0), allow_download=True)


## 3. Configuración sin fuga de test

Usa una carpeta que contenga **solo identidades de train**, con `id_x/edad[_repeticion].jpg`. Si usas la raíz completa, cambia `train_only=False` y proporciona `split_path` con columnas `person_id,split`. El test se utiliza exclusivamente para excluir IDs, nunca para ajustar referencias.

FFHQ requiere imágenes locales y `ffhq_aging_labels.csv` oficial. No se sustituye por Kaggle/FG-NET. Fotos de edades cercanas pero distintas son necesarias para ICC; sin ellas no se inventa confiabilidad.


In [ ]:
reference_config = ReferenceConfig(
    colombia_root=COLOMBIA_TRAIN, ffhq_aging_root=FFHQ_AGING,
    output_dir=REFERENCE_DIR, train_only=True,
    test_manifest_path=TEST_MANIFEST,
    # split_path="/content/drive/MyDrive/dataset_def/splits.csv",
    features=reference_backend["feature_config"],
    ffhq_min_conf=0.6, ffhq_max_per_group=500,
    min_icc=0.5, min_retest_pairs=10, seed=2026, resume=True,
)


In [ ]:
from src.aging_bias.references import load_colombia_manifest
from itertools import combinations
train = load_colombia_manifest(reference_config)
near_pairs = sum(abs(float(a)-float(b)) <= 1 for _, g in train.groupby("identity_id") for a,b in combinations(g.age,2))
print(f"Train: {train.identity_id.nunique()} identidades, {len(train)} imágenes, {near_pairs} pares cercanos potenciales")
assert near_pairs >= reference_config.min_retest_pairs, "No hay suficientes retests; revisar train antes de extraer atributos"


In [ ]:
references = build_reference_bundle(
    reference_config, feature_backend=reference_backend["feature_backend"],
    landmark_backend=reference_backend["landmark_backend"],
)
display(pd.read_csv(REFERENCE_DIR / "attribute_reliability.csv"))
print("Atributos activos:", references.attributes)


## 4. ¿Las poblaciones envejecen distinto?

Bootstrap por identidad en Colombia y por imagen dentro del grupo de edad en FFHQ. Revisa soporte, intervalos e identificación antes de evaluar modelos. El problema edad-período-cohorte no desaparece por compartir referencias.


In [ ]:
population_table = describe_population_aging(
    references, transitions=[(20,50),(30,60),(40,70)],
    n_boot=1000, seed=2026, output_dir=REFERENCE_DIR,
)
display(population_table)
print(population_table.attrs["identification"])
from IPython.display import Image, display
display(Image(filename=str(REFERENCE_DIR / "population_aging.png")))


## Salida

`references.npz` + `references.json`, confiabilidad/exclusiones, features de train/proxy y contrastes poblacionales. Conserva toda la carpeta. La PCA DINOv2 es opcional: `src.aging_bias.embedding_features.EmbeddingFeatureBackend`, ajustada **solo** con crops comunes de train + proxy y con referencias separadas.
